# Find similarities and duplicates
## Text Embeddings
Use the [SentenceTransformers (SBERT)](https://sbert.net/) to create text embeddings from the following marc fields:
```
- id: 001,  
- title: 245$a, 245$b, 245$p, 245$f  
- pagination: 300$a  
- publication year: 008[7:11], 008[11:15], 264$c, 260$c
- vernacular title: linked fields of 245
- vernacular author: linked fields of 100, 110, 111
- context title: 505$t
- edition: 250$a
- publisher name: 264$b, 260$b`
```
As a first step we use the trained model `model_name="all-MiniLM-L6-v2"`. 

We parse the marcxml file and create a string from the values of the marc fields. We use this string to create text embeddings.We're saving the text embedding in a new json file that includes the text_embedding field and the record id. 

In [ ]:
!pip3 install pymarc
!pip3 install sentence-transformers
!pip3 install pandas

In [ ]:
import tarfile
import pymarc
import glob
import os, tarfile, re, json
from pymarc import parse_xml_to_array
import string
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from marcxml_embedding_parser import MARCXMLEmbeddingParser

In [ ]:
marcxml_dir = "data_marcxml"

In [ ]:
parser = MARCXMLEmbeddingParser();

In [ ]:
parser.parse_scsb_update_files(input_dir=marcxml_dir, batch_size=10000)

## PCA (Principal Component Analysis)

PCA enables dimensionality reduction. Text embeddings are represented as vectors in a multi-dimensional space. We employ Principal Component Analysis (PCA) to compute the number of components required to explain 95% of the variance within the data. PCA enables dimensionality reduction.

To achieve this, we utilize Incremental Principal Component Analysis (IPCA), which allows for efficient processing of large datasets by incrementally updating the PCA model. We load the data in batches, specifically 10,000 records per file, to ensure manageable memory usage and to calculate the Explained Variance Ratio and the Cumulative Explained Variance.

For similarity detection and duplicate identification, we apply the Euclidean distance metric. The threshold for determining similarity and duplication is adjusted based on the calculated distance ranges. The smaller we set the Euclidean distance, the more identical records we expect to identify.